### ЗАДАЧА: Синхронизация каталога поставщиков

Команда e-commerce каждый вечер получает выгрузку от нескольких поставщиков.
Нужно подготовить два результата:
- партнерский `JSON`-отчет для витрины и внешней интеграции,
- полный внутренний `pickle`-снимок для отладки, повторной загрузки и аудита.

НЕОБХОДИМО:

1. Преобразовать строки из `rows` в список словарей `products_data`.
2. Для каждого товара привести типы:
   - `stock` -> `int`
   - `price` -> `float`
   - `discount_percent` -> `int`
   - `is_active` -> `bool`
   - `tags` -> `list[str]`
3. Для каждого товара вычислить поля:
   - `final_price`
   - `stock_value`
4. Построить словарь `supplier_summary`, где по каждому поставщику будет:
   - `products_count`
   - `active_products_count`
   - `total_stock`
   - `inventory_value`
5. Построить словарь `category_summary`, где по каждой категории будет:
   - `products_count`
   - `active_products_count`
   - `min_final_price`
   - `max_final_price`
6. Собрать список `partner_feed` только из активных товаров.
7. Для каждого товара в `partner_feed` оставить только поля:
   - `sku`
   - `title`
   - `category`
   - `supplier`
   - `final_price`
   - `stock`
   - `tags`
8. Собрать список `attention_items` для внутренней команды.
   Товар нужно добавить в `attention_items`, если:
   - `stock <= 3`,
   - или `discount_percent >= 40`,
   - или `is_active == False`.
9. Для `attention_items` оставить поля:
   - `sku`
   - `title`
   - `supplier`
   - `stock`
   - `discount_percent`
   - `is_active`
10. Отсортировать `partner_feed` по `final_price` по возрастанию.
11. Отсортировать `attention_items` по `stock` по возрастанию.
12. Собрать `public_report` со структурой:
   - `report_name`
   - `products_count`
   - `partner_feed`
   - `supplier_summary`
   - `category_summary`
13. Сохранить `public_report` в файл `partner_catalog_report.json`.
14. Сохранить внутренний снимок в файл `partner_catalog_snapshot.pkl`.
15. Прочитать оба файла обратно и вывести результаты.


In [6]:
import json
import pickle


# row format: sku|title|category|supplier|stock|price|discount_percent|is_active|tags
rows = [
    "KB-001|Mechanical Keyboard|electronics|KeyPro|12|8900.00|10|true|gaming,peripherals",
    "MS-210|Wireless Mouse|electronics|KeyPro|2|3200.00|5|true|wireless,peripherals",
    "CHR-77|Office Chair|furniture|HomeSpace|5|15990.00|25|true|office,comfort",
    "DSK-15|Standing Desk|furniture|HomeSpace|1|42990.00|40|true|office,premium",
    "MUG-09|Thermo Mug|lifestyle|DailyCo|18|990.00|0|true|kitchen,gift",
    "LMP-44|Desk Lamp|lifestyle|DailyCo|0|2490.00|15|false|home,office",
    "MON-32|4K Monitor|electronics|VisionTech|4|32990.00|12|true|display,premium",
    "WEB-12|Webcam FullHD|electronics|VisionTech|3|5400.00|45|true|streaming,peripherals",
]

products_data = []
supplier_summary = {}
category_summary = {}
partner_feed = []
attention_items = []

for row in rows:
    sku, title, category, supplier, stock_raw, price_raw, discount_raw, is_active_raw, tags_raw = row.split("|")

    # TODO 1: преобразуйте stock_raw в int и сохраните в stock
    # TODO 2: преобразуйте price_raw в float и сохраните в price
    # TODO 3: преобразуйте discount_raw в int и сохраните в discount_percent
    # TODO 4: преобразуйте is_active_raw в bool и сохраните в is_active
    # TODO 5: разделите tags_raw по ',' и сохраните в tags
    stock = int(stock_raw)
    price = float(price_raw)
    discount_percent = int(discount_raw)
    is_active = True if is_active_raw == 'true' else False
    tags = tags_raw.split(',')
    # TODO 6: вычислите final_price по формуле:
    #   price * (1 - discount_percent / 100)
    final_price = price * (1 - discount_percent / 100)

    # TODO 7: вычислите stock_value = stock * final_price
    stock_value = stock * final_price

    product = {
        "sku": sku,
        "title": title,
        "category": category,
        "supplier": supplier,
        # TODO 8: добавьте stock
        # TODO 9: добавьте price
        # TODO 10: добавьте discount_percent
        # TODO 11: добавьте is_active
        # TODO 12: добавьте tags
        # TODO 13: добавьте final_price
        # TODO 14: добавьте stock_value
        'stock': stock,
        'price': price,
        'discount_percent': discount_percent,
        'is_active': is_active,
        'tags': tags,
        "final_price": final_price, 
        'stock_value': stock_value
    }

    products_data.append(product)

    # TODO 15: если supplier еще отсутствует в supplier_summary,
    # создайте для него словарь с полями:
    #   products_count -> 0
    #   active_products_count -> 0
    #   total_stock -> 0
    #   inventory_value -> 0.0
    supplier_summary.setdefault(supplier, {'products_count': 0, 'active_products_count': 0, 'total_stock': 0, 'inventory_value': 0})
    # TODO 16: увеличьте products_count на 1
    # TODO 17: увеличьте total_stock на stock
    # TODO 18: прибавьте stock_value к inventory_value
    # TODO 19: если товар активен, увеличьте active_products_count на 1
    y = supplier_summary[supplier]
    y['products_count'] += 1
    y['total_stock'] += stock
    y['inventory_value'] += stock_value
    if is_active == True:
        y['active_products_count'] += 1
    # TODO 20: если category еще отсутствует в category_summary,
    # создайте для нее словарь с полями:
    #   products_count -> 0
    #   active_products_count -> 0
    #   min_final_price -> final_price
    #   max_final_price -> final_price
    category_summary.setdefault(category, {'products_count': 0, 'active_products_count': 0, 'min_final_price': final_price, 'max_final_price': final_price})

    # TODO 21: увеличьте category_summary[category]['products_count'] на 1
    # TODO 22: если товар активен, увеличьте active_products_count на 1
    category_summary[category]['products_count'] += 1
    if is_active:
        category_summary[category]['active_products_count'] += 1
    # TODO 23: обновите min_final_price, если final_price меньше текущего
    # TODO 24: обновите max_final_price, если final_price больше текущего
    if final_price > category_summary[category]['max_final_price']:
        category_summary[category]['max_final_price'] = final_price
    elif final_price < category_summary[category]['min_final_price']:
        category_summary[category]['min_final_price'] = final_price
    # TODO 25: если товар активен, добавьте в partner_feed словарь с нужными полями
    if is_active:
        partner_feed.append({'sku': sku, 'title': title, 'category': category, 'supplier': supplier, 'final_price': final_price, 'stock': stock, 'tags': tags})
    # TODO 26: если товар требует внимания,
    # добавьте его в attention_items только с нужными полями
    if stock <= 3 or discount_percent >= 40 or is_active == False:
        attention_items.append({'sku': sku, 'title': title, 'supplier': supplier, "discount_percent": discount_percent, 'is_active': is_active, 'stock': stock})

# TODO 27: отсортируйте partner_feed по final_price по возрастанию
# TODO 28: отсортируйте attention_items по stock по возрастанию
partner_feed.sort(key=lambda x: x['final_price'])
attention_items.sort(key=lambda x: x['stock'])

# TODO 29: округлите inventory_value у всех поставщиков до 2 знаков
# TODO 30: округлите min_final_price и max_final_price у всех категорий до 2 знаков
for val in supplier_summary.values():
    val['inventory_value'] = round(val['inventory_value'], 2)

for val in category_summary.values():
    val['min_final_price'], val['max_final_price'] = round(val['min_final_price'], 2), round(val['max_final_price'], 2)


public_report = {
    "report_name": "partner_catalog_sync",
    "products_count": len(products_data),
    # TODO 31: добавьте partner_feed
    # TODO 32: добавьте supplier_summary
    # TODO 33: добавьте category_summary
    'partner_feed': partner_feed,
    'supplier_summary': supplier_summary, 
    'category_summary': category_summary
}

snapshot = {
    "rows": rows,
    "products_data": products_data,
    "partner_feed": partner_feed,
    "supplier_summary": supplier_summary,
    "category_summary": category_summary,
    "attention_items": attention_items,
}

# TODO 34: сохраните public_report в partner_catalog_report.json через json.dump
#   используйте ensure_ascii=False и indent=2
with open('j.json', 'w', encoding='utf-8') as f:
    json.dump(public_report, f, ensure_ascii=False, indent=2)
# TODO 35: сохраните snapshot в partner_catalog_snapshot.pkl через pickle.dump
with open('p.pkl', 'wb') as f:
    pickle.dump(snapshot, f)
# TODO 36: прочитайте partner_catalog_report.json в loaded_report через json.load
# TODO 37: прочитайте partner_catalog_snapshot.pkl в loaded_snapshot через pickle.load
with open('j.json', 'r', encoding='utf-8') as f:
    loaded_report = json.load(f)

with open('p.pkl', 'rb',) as f:
    loaded_snapshot = pickle.load(f)

print("Полный каталог:")
print(products_data)
print()

print("Партнерский отчет:")
print(public_report)
print()

print("Данные из JSON:")
print(loaded_report)
print()

print("Данные из pickle:")
print(loaded_snapshot)
print()

print("Товары, требующие внимания:")
print(attention_items)


Полный каталог:
[{'sku': 'KB-001', 'title': 'Mechanical Keyboard', 'category': 'electronics', 'supplier': 'KeyPro', 'stock': 12, 'price': 8900.0, 'discount_percent': 10, 'is_active': True, 'tags': ['gaming', 'peripherals'], 'final_price': 8010.0, 'stock_value': 96120.0}, {'sku': 'MS-210', 'title': 'Wireless Mouse', 'category': 'electronics', 'supplier': 'KeyPro', 'stock': 2, 'price': 3200.0, 'discount_percent': 5, 'is_active': True, 'tags': ['wireless', 'peripherals'], 'final_price': 3040.0, 'stock_value': 6080.0}, {'sku': 'CHR-77', 'title': 'Office Chair', 'category': 'furniture', 'supplier': 'HomeSpace', 'stock': 5, 'price': 15990.0, 'discount_percent': 25, 'is_active': True, 'tags': ['office', 'comfort'], 'final_price': 11992.5, 'stock_value': 59962.5}, {'sku': 'DSK-15', 'title': 'Standing Desk', 'category': 'furniture', 'supplier': 'HomeSpace', 'stock': 1, 'price': 42990.0, 'discount_percent': 40, 'is_active': True, 'tags': ['office', 'premium'], 'final_price': 25794.0, 'stock_valu